# IE492 — Phase 2.2: Beceri Bazlı Eşleştirme (Heterojen Terkli Model)

**Model Özeti:**
- Oyuncular Poisson($\lambda$) süreciyle gelir.
- $p$ olasılıkla **İyi (Good)**, $1-p$ olasılıkla **Kötü (Bad)** beceri seviyesindedir.
- **Heterojen Sabır Süresi:**
  - İyi oyuncular: $\text{Exp}(\theta_G)$
  - Kötü oyuncular: $\text{Exp}(\theta_B)$
- **Homojen Öncelik:** İyiler iyilerle ($G-G$), kötüler kötülerle ($B-B$) eşleşir.
- **Çapraz Eşleşme Eşiği ($\tau$):** Kendi seviyesinden oyuncu bulamayan bir oyuncu $\tau$ süre bekledikten sonra "esnek" hale gelir ve karşı seviyedeki oyuncuyla eşleşebilir ($G-B$).
- **Terk Etkileşimi:** Sabır süresi $< \tau$ ise oyuncu cross-match göremeden ayrılır. Sabır $\ge \tau$ ise esnek havuza girer, yine de eşleşemezse sabrı dolunca terk eder.

In [ ]:
import heapq
from collections import deque
import numpy as np
import matplotlib.pyplot as plt

GOOD, BAD = 0, 1
TIER_NAME = {GOOD: "İyi (Good)", BAD: "Kötü (Bad)"}

WAITING = 0     # Kendi sınıfını bekliyor (< tau)
FLEXIBLE = 1    # tau süresini aştı, cross-match havuzunda
MATCHED = 2     # Eşleşti
ABANDONED = 3   # Sabrı bitti, terk etti

EVT_ARRIVAL = 1
EVT_TIMEOUT = 2
EVT_ABANDON = 3

MATCH_HOMOGENEOUS = 0  # G-G veya B-B
MATCH_CROSS = 1        # G-B

## Simülasyon Fonksiyonları

In [ ]:
def simulate_phase2_2(lam=1.0, p=0.6, tau=3.0, theta_g=0.08, theta_b=0.04,
                      T=50_000, warmup=1_000, seed=42):
    rng = np.random.default_rng(seed)
    theta_map = {GOOD: theta_g, BAD: theta_b}

    arrival, tier, end, status, partner_id, match_type = [], [], [], [], [], []
    queues = {GOOD: deque(), BAD: deque()}
    n_waiting = {GOOD: 0, BAD: 0}
    events = []
    area = {GOOD: 0.0, BAD: 0.0}
    last_t = 0.0
    trace_t, trace_q_good, trace_q_bad, trace_q_total = [0.0], [0], [0], [0]

    def advance(to):
        nonlocal last_t
        start = max(last_t, warmup)
        if to > start:
            for c in (GOOD, BAD):
                area[c] += n_waiting[c] * (to - start)
        last_t = to

    def record_trace(t):
        trace_t.append(t)
        trace_q_good.append(n_waiting[GOOD])
        trace_q_bad.append(n_waiting[BAD])
        trace_q_total.append(n_waiting[GOOD] + n_waiting[BAD])

    t_first = rng.exponential(1.0 / lam)
    heapq.heappush(events, (t_first, EVT_ARRIVAL, -1))

    while events:
        evt_t, evt_type, pid = heapq.heappop(events)
        if evt_t > T:
            advance(T)
            break
        advance(evt_t)

        if evt_type == EVT_ARRIVAL:
            t_next = evt_t + rng.exponential(1.0 / lam)
            if t_next <= T:
                heapq.heappush(events, (t_next, EVT_ARRIVAL, -1))

            c = GOOD if rng.random() < p else BAD
            other = 1 - c
            curr_id = len(arrival)
            arrival.append(evt_t); tier.append(c); end.append(np.nan)
            status.append(WAITING); partner_id.append(-1); match_type.append(-1)

            # 1. Kendi sınıfından bekleyen var mı?
            matched_partner = None
            while queues[c]:
                cand = queues[c].popleft()
                if status[cand] in (WAITING, FLEXIBLE):
                    matched_partner = cand
                    break

            if matched_partner is not None:
                status[matched_partner] = MATCHED; end[matched_partner] = evt_t
                partner_id[matched_partner] = curr_id; match_type[matched_partner] = MATCH_HOMOGENEOUS
                status[curr_id] = MATCHED; end[curr_id] = evt_t
                partner_id[curr_id] = matched_partner; match_type[curr_id] = MATCH_HOMOGENEOUS
                n_waiting[c] -= 1
                record_trace(evt_t)
                continue

            # 2. Karşı sınıfta FLEXIBLE var mı?
            cross_partner = None
            for cand in queues[other]:
                if status[cand] == FLEXIBLE:
                    cross_partner = cand
                    break

            if cross_partner is not None:
                queues[other].remove(cross_partner)
                status[cross_partner] = MATCHED; end[cross_partner] = evt_t
                partner_id[cross_partner] = curr_id; match_type[cross_partner] = MATCH_CROSS
                status[curr_id] = MATCHED; end[curr_id] = evt_t
                partner_id[curr_id] = cross_partner; match_type[curr_id] = MATCH_CROSS
                n_waiting[other] -= 1
                record_trace(evt_t)
                continue

            # Eşleşme yok -> Kuyruğa ekle
            queues[c].append(curr_id)
            n_waiting[c] += 1
            heapq.heappush(events, (evt_t + tau, EVT_TIMEOUT, curr_id))
            th = theta_map[c]
            if th > 0:
                patience = rng.exponential(1.0 / th)
                heapq.heappush(events, (evt_t + patience, EVT_ABANDON, curr_id))
            record_trace(evt_t)

        elif evt_type == EVT_TIMEOUT:
            if status[pid] == WAITING:
                status[pid] = FLEXIBLE
                c = tier[pid]; other = 1 - c
                cross_partner = None
                while queues[other]:
                    cand = queues[other].popleft()
                    if status[cand] in (WAITING, FLEXIBLE):
                        cross_partner = cand
                        break

                if cross_partner is not None:
                    queues[c].remove(pid)
                    status[pid] = MATCHED; end[pid] = evt_t
                    partner_id[pid] = cross_partner; match_type[pid] = MATCH_CROSS
                    status[cross_partner] = MATCHED; end[cross_partner] = evt_t
                    partner_id[cross_partner] = pid; match_type[cross_partner] = MATCH_CROSS
                    n_waiting[c] -= 1; n_waiting[other] -= 1
                    record_trace(evt_t)

        elif evt_type == EVT_ABANDON:
            if status[pid] in (WAITING, FLEXIBLE):
                c = tier[pid]
                status[pid] = ABANDONED
                end[pid] = evt_t
                queues[c].remove(pid)
                n_waiting[c] -= 1
                record_trace(evt_t)

    players = {
        "arrival": np.array(arrival), "tier": np.array(tier),
        "end": np.array(end), "status": np.array(status),
        "partner_id": np.array(partner_id), "match_type": np.array(match_type),
    }
    span = T - warmup
    avg_queue = {
        GOOD: area[GOOD] / span,
        BAD: area[BAD] / span,
        "total": (area[GOOD] + area[BAD]) / span
    }
    trace = {
        "t": np.array(trace_t),
        "good": np.array(trace_q_good),
        "bad": np.array(trace_q_bad),
        "total": np.array(trace_q_total),
    }
    return players, avg_queue, trace

def compute_kpis_phase2_2(players, avg_queue, warmup=1_000):
    kpis = {}
    for c in (GOOD, BAD):
        sel = (players["tier"] == c) & (players["arrival"] >= warmup) & (players["status"] != WAITING) & (players["status"] != FLEXIBLE)
        st = players["status"][sel]
        w = players["end"][sel] - players["arrival"][sel]
        w_matched = w[st == MATCHED]
        mt = players["match_type"][sel]

        kpis[TIER_NAME[c]] = {
            "oyuncu sayisi": int(sel.sum()),
            "eslesme orani (%)": float((st == MATCHED).mean() * 100.0) if len(st) > 0 else 0.0,
            "terk orani (%)": float((st == ABANDONED).mean() * 100.0) if len(st) > 0 else 0.0,
            "ort. bekleme (tumu)": float(w.mean()) if len(w) > 0 else 0.0,
            "ort. bekleme (eslesen)": float(w_matched.mean()) if len(w_matched) > 0 else 0.0,
            "medyan bekleme (eslesen)": float(np.median(w_matched)) if len(w_matched) > 0 else 0.0,
            "p90 bekleme (eslesen)": float(np.percentile(w_matched, 90)) if len(w_matched) > 0 else 0.0,
            "ort. kuyruk uzunlugu": avg_queue[c],
            "homojen mac sayisi": int((mt == MATCH_HOMOGENEOUS).sum()),
            "capraz mac sayisi": int((mt == MATCH_CROSS).sum()),
        }

    sel_all = (players["arrival"] >= warmup) & (players["status"].isin([MATCHED, ABANDONED]))
    st_all = players["status"][sel_all]
    w_all = players["end"][sel_all] - players["arrival"][sel_all]
    w_matched_all = w_all[st_all == MATCHED]
    mt_matched = players["match_type"][sel_all & (players["status"] == MATCHED)]

    total_matched = len(w_matched_all)
    cross_matched = int((mt_matched == MATCH_CROSS).sum())
    cross_ratio = cross_matched / total_matched if total_matched > 0 else 0.0

    kpis["genel"] = {
        "toplam oyuncu": len(st_all),
        "genel eslesme orani (%)": float((st_all == MATCHED).mean() * 100.0) if len(st_all) > 0 else 0.0,
        "genel terk orani (%)": float((st_all == ABANDONED).mean() * 100.0) if len(st_all) > 0 else 0.0,
        "ort. bekleme (eslesen)": float(w_matched_all.mean()) if len(w_matched_all) > 0 else 0.0,
        "ort. toplam kuyruk": avg_queue["total"],
        "capraz eslesme orani (%)": cross_ratio * 100.0,
        "homojen eslesme orani (%)": (1.0 - cross_ratio) * 100.0,
    }
    return kpis

def print_kpis_phase2_2(kpis):
    print("=" * 64)
    print(f"{'KPI':<30}{'İyi (Good)':>16}{'Kötü (Bad)':>16}")
    print("-" * 64)
    for k in ["oyuncu sayisi", "eslesme orani (%)", "terk orani (%)",
              "ort. bekleme (tumu)", "ort. bekleme (eslesen)",
              "medyan bekleme (eslesen)", "p90 bekleme (eslesen)",
              "ort. kuyruk uzunlugu", "capraz mac sayisi"]:
        print(f"{k:<30}{kpis[TIER_NAME[GOOD]][k]:>16.3f}{kpis[TIER_NAME[BAD]][k]:>16.3f}")
    print("-" * 64)
    print(f"Genel Eşleşme Oranı           : %{kpis['genel']['genel eslesme orani (%)']:>9.2f}")
    print(f"Genel Terk Oranı              : %{kpis['genel']['genel terk orani (%)']:>9.2f}")
    print(f"Genel Ort. Bekleme (Eşleşen)  : {kpis['genel']['ort. bekleme (eslesen)']:>10.3f}")
    print(f"Genel Ortalama Toplam Kuyruk  : {kpis['genel']['ort. toplam kuyruk']:>10.3f}")
    print(f"Çapraz Eşleşme Oranı (G-B)    : %{kpis['genel']['capraz eslesme orani (%)']:>9.2f}")
    print(f"Homojen Eşleşme Oranı (G-G,B-B): %{kpis['genel']['homojen eslesme orani (%)']:>8.2f}")
    print("=" * 64)

## Tek Senaryo Koşumu
Parametreler: $\lambda=1.0, p=0.6, \tau=3.0, \theta_G=0.08, \theta_B=0.04$.

In [ ]:
LAM = 1.0
P = 0.6
TAU = 3.0
THETA_G = 0.08
THETA_B = 0.04
T = 50_000
WARMUP = 1_000
SEED = 42

players, avg_queue, trace = simulate_phase2_2(
    lam=LAM, p=P, tau=TAU, theta_g=THETA_G, theta_b=THETA_B,
    T=T, warmup=WARMUP, seed=SEED
)
kpis = compute_kpis_phase2_2(players, avg_queue, warmup=WARMUP)
print_kpis_phase2_2(kpis)

## Grafikler: Kuyruk Seyri ve Terk / Eşleşme Dağılımı

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 4))
tt = trace["t"]
window = (tt >= WARMUP) & (tt <= WARMUP + 2000)

# 1. Sınıf bazında kuyruklar
axs[0].step(tt[window], trace["good"][window], label="İyi Kuyruğu", alpha=0.8)
axs[0].step(tt[window], trace["bad"][window], label="Kötü Kuyruğu", alpha=0.8)
axs[0].set_xlabel("Zaman")
axs[0].set_ylabel("Bekleyen Sayısı")
axs[0].set_title("Kuyruk Durumu (Pencere)")
axs[0].legend()

# 2. Toplam kuyruk
axs[1].step(tt[window], trace["total"][window], color="tab:purple", label="Anlık Toplam")
axs[1].axhline(kpis["genel"]["ort. toplam kuyruk"], color="red", linestyle=":", label=f"Ort. = {kpis['genel']['ort. toplam kuyruk']:.2f}")
axs[1].set_xlabel("Zaman")
axs[1].set_ylabel("Toplam Kuyruk")
axs[1].set_title("Toplam Kuyruk Uzunluğu")
axs[1].legend()

# 3. Eşleşme vs Terk Bekleme Dağılımı
sel_m = (players["status"] == MATCHED) & (players["arrival"] >= WARMUP)
sel_a = (players["status"] == ABANDONED) & (players["arrival"] >= WARMUP)
wm = players["end"][sel_m] - players["arrival"][sel_m]
wa = players["end"][sel_a] - players["arrival"][sel_a]

axs[2].hist(wm[wm > 0], bins=40, alpha=0.6, label="Eşleşenler", density=True)
axs[2].hist(wa, bins=40, alpha=0.6, label="Terk Edenler", density=True, color="crimson")
axs[2].axvline(TAU, color="black", linestyle="--", label=f"$\\tau={TAU}$")
axs[2].set_xlabel("Bekleme Süresi")
axs[2].set_ylabel("Yoğunluk")
axs[2].set_title("Eşleşme vs Terk Dağılımı")
axs[2].legend()
plt.tight_layout()
plt.show()

## $\tau$ Duyarlılık Analizi: Terk Oranını Düşürme Gücü
Cross-match eşiği $\tau$'nun terk oranını ve maç dengesini nasıl etkilediği.

In [ ]:
TAU_GRID = [1.0, 2.0, 3.0, 5.0, 8.0, 15.0, 25.0]
res2 = []

for tau_val in TAU_GRID:
    pl, aq, _ = simulate_phase2_2(
        lam=LAM, p=P, tau=tau_val, theta_g=THETA_G, theta_b=THETA_B,
        T=30_000, warmup=1_000, seed=SEED
    )
    rk = compute_kpis_phase2_2(pl, aq, warmup=1_000)
    res2.append({
        "tau": tau_val,
        "terk_genel": rk["genel"]["genel terk orani (%)"],
        "terk_good": rk[TIER_NAME[GOOD]]["terk orani (%)"],
        "terk_bad": rk[TIER_NAME[BAD]]["terk orani (%)"],
        "w_matched": rk["genel"]["ort. bekleme (eslesen)"],
        "cross_ratio": rk["genel"]["capraz eslesme orani (%)"],
    })

print(f"{'tau':>6} | {'Terk Genel (%)':>16} | {'Terk İyi (%)':>14} | {'Terk Kötü (%)':>14} | {'Çapraz Maç (%)':>16}")
print("-" * 75)
for r in res2:
    print(f"{r['tau']:>6.1f} | {r['terk_genel']:>15.2f}% | {r['terk_good']:>13.2f}% | {r['terk_bad']:>13.2f}% | {r['cross_ratio']:>15.2f}%")

fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))
taus = [r["tau"] for r in res2]
axs[0].plot(taus, [r["terk_genel"] for r in res2], marker="o", label="Genel Terk (%)", color="red")
axs[0].plot(taus, [r["terk_good"] for r in res2], marker="^", label="İyi Terk (%)")
axs[0].plot(taus, [r["terk_bad"] for r in res2], marker="v", label="Kötü Terk (%)")
axs[0].set_xlabel("Çapraz Eşleşme Eşiği ($\\tau$)")
axs[0].set_ylabel("Terk Oranı (%)")
axs[0].set_title("$\\tau$ vs Terk Oranı")
axs[0].legend()
axs[0].grid(True, alpha=0.3)

axs[1].plot([r["cross_ratio"] for r in res2], [r["terk_genel"] for r in res2], marker="D", color="darkgreen")
for r in res2:
    axs[1].annotate(f"$\\tau={r['tau']}$", (r["cross_ratio"], r["terk_genel"]), textcoords="offset points", xytext=(5, 5))
axs[1].set_xlabel("Çapraz Eşleşme Oranı (%)")
axs[1].set_ylabel("Genel Terk Oranı (%)")
axs[1].set_title("Trade-off: Terk Oranı vs Çapraz Maç Oranı")
axs[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()